# Crea tu primer agente con Claude Agent SDK

Un notebook con tres agentes. Cada celda suma un primitivo.

- **El Parcero Crítico**: clona un repo público y lo roastea.
- **Buscador de vivienda**: encuentra avisos reales y los evalúa.
- **El Monitor**: te explica un paper como el compañero que sí lo leyó.

Cada agente vive en dos lugares: un archivo `agent_<nombre>.py` (el código) y una carpeta `workspace_<nombre>/` (su casa: lo que ve, lo que carga, donde escribe).

## 0. Antes de empezar

- Tienes Claude Code instalado y con sesión iniciada: `claude` → `/login`.
- **No** tienes `ANTHROPIC_API_KEY` en el entorno. Si está, el SDK cobra por la API en vez de usar tu suscripción.
- En un notebook se escribe `await`, nunca `asyncio.run()`: el notebook ya tiene su propio loop corriendo.

Sin suscripción de Claude, cambia `PROVEEDOR` en la celda de abajo: `"openrouter"` (modelos gratis), `"opencode"` (OpenCode Zen, gratis) o `"codex"` (suscripción de ChatGPT vía LiteLLM). Cada uno tiene su paso a paso en `docs/proveedores.md`. El Monitor (sección 5) necesita `"claude"`.

In [ ]:
import os, subprocess
print("claude:", subprocess.run(["claude", "--version"], capture_output=True, text=True).stdout.strip())
print("ANTHROPIC_API_KEY:", "puesta (quítala para usar tu suscripción)" if os.environ.get("ANTHROPIC_API_KEY") else "no está, bien")

In [ ]:
from proveedores import proveedor

PROVEEDOR = "claude"       # "claude" · "openrouter" · "opencode" · "codex"  (docs/proveedores.md)

MODELO, ENV = proveedor(PROVEEDOR)
os.environ.update(ENV)                        # lo heredan todos los agentes de este notebook
os.environ["TALLER_PROVEEDOR"] = PROVEEDOR    # y los agent_*.py
print("cerebro:", PROVEEDOR, "→", MODELO)

## 1. El enigma

Todo lo que vas a ver aquí ya lo hace Claude Code en tu terminal: leer archivos, correr comandos, buscar en internet, usar skills y MCPs.

Entonces, **¿para qué un SDK?**

Guarda la pregunta. La respondemos al final, con los tres agentes ya corriendo.

## 2. El concepto: un agente es un modelo más un harness

- **Modelo**: el que piensa. Recibe texto, devuelve texto. No puede tocar nada.
- **Harness**: todo lo demás. Las herramientas, el ciclo que las llama, la memoria, los permisos.
- **Agente** = modelo + harness.

Claude Code es un harness. El **Agent SDK** es ese mismo harness, pero como librería: lo llamas desde Python con una función, `query()`, y le pasas tu configuración.

Los primitivos que vamos a ir sumando, uno por celda:

| # | Primitivo | Qué le da al agente |
|---|---|---|
| 1 | `query()` + `system_prompt` | una personalidad y una vuelta |
| 2 | `cwd` + tools nativas | manos: leer, correr comandos |
| 3 | el stream de mensajes | ver cada paso del ciclo |
| 4 | `setting_sources` + skill | un procedimiento que carga solo cuando hace falta |
| 5 | hooks | un portero que dice no |
| 6 | sub-agentes | ayudantes que trabajan en paralelo |
| 7 | streaming | texto letra por letra |
| 8 | buscadores por MCP + `max_budget_usd` | internet, con tope de gasto |
| 9 | `resume` | seguir una conversación |
| 10 | MCP externo | herramientas de otros |
| 11 | dos `query()` a la vez | orquestar agentes desde código |

## 3. El Parcero Crítico, un primitivo a la vez

### 3.1 · `query()` + `system_prompt`: una personalidad, una vuelta

Lo mínimo. El modelo recibe quién es y una pregunta. Sin herramientas: solo puede hablar.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage

SYSTEM_PROMPT = """Eres El Parcero Crítico: revisas código ajeno con humor seco y criterio técnico.
Hablas en español colombiano, tutea, con frases cortas. Te burlas del código, nunca de quien lo escribió.

Rosteas el código que te ingresen

"""

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,   # quién es
    tools=[],                      # sin herramientas
    max_turns=1,                   # una sola vuelta
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = """def hola():
    print("Hola, muuuundo!")"""

async for mensaje in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(mensaje, AssistantMessage):
        for bloque in mensaje.content:
            if isinstance(bloque, TextBlock): print(bloque.text)
    elif isinstance(mensaje, ResultMessage):
        print(f"\n[fin] {mensaje.num_turns} vueltas · USD {mensaje.total_cost_usd:.4f}")

Dos cosas para mirar:

- `query()` devuelve mensajes uno por uno. El último es `ResultMessage`: el recibo (vueltas y costo).
- Con `max_turns=1` y `tools=[]` esto es un chat. Todavía no es un agente.

### 3.2 · `cwd` + tools nativas: manos

Le damos una casa (`cwd`) y una herramienta (`Bash`). Ahora puede hacer algo en el mundo: clonar un repo.

`allowed_tools` dice qué herramientas se aprueban sin preguntar. `setting_sources=[]` dice: no leas configuración de disco todavía.

In [ ]:
from pathlib import Path
from claude_agent_sdk import ToolUseBlock

WS = Path("workspace_parcero").resolve()      # su casa
(WS / "repos").mkdir(exist_ok=True)

SYSTEM_PROMPT = f"""# Quién eres
Eres El Parcero Crítico: revisas código ajeno con humor seco y criterio técnico.
Hablas en español colombiano sin groserías, de tú, con frases cortas.
Te burlas del código, nunca de quien lo escribió.

# Dónde trabajas
- Tu carpeta es `{WS}`. Los repos se clonan en `{WS}/repos/`.
- Usa rutas absolutas. No uses `cd`: el shell recuerda el `cd` entre órdenes y te vas a perder.
- Si el repo ya está en `repos/`, no lo clones otra vez.

clona el repo que te indiquen y haz el rosteo

"""

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,
    cwd=str(WS),
    allowed_tools=["Bash"],        # una sola mano
    setting_sources=[],
    max_turns=10,
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = "https://github.com/JairoTorregrosa/crea-tu-web"

async for mensaje in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(mensaje, AssistantMessage):
        for bloque in mensaje.content:
            if isinstance(bloque, ToolUseBlock): print(f"[usa] {bloque.name}: {bloque.input.get('command', '')}")
            if isinstance(bloque, TextBlock): print(bloque.text)
    elif isinstance(mensaje, ResultMessage):
        print(f"\n[fin] {mensaje.num_turns} vueltas · USD {mensaje.total_cost_usd:.3f}")

Cada `[usa]` es una vuelta del ciclo: el modelo pide, el harness ejecuta, el resultado vuelve al modelo, y el modelo decide qué sigue. Eso es un agente.

### 3.3 · El stream: ver cada paso del ciclo

Esta vez imprimimos **todo** lo que llega, por tipo de mensaje. Son cuatro:

- `SystemMessage` (`init`): con qué arrancó.
- `AssistantMessage`: el modelo habla o pide una herramienta.
- `UserMessage` con `ToolResultBlock`: la herramienta responde.
- `ResultMessage`: el recibo.

Es el mismo juego que hicimos con personas: cada mensaje es un papel que pasa de mano en mano.

In [ ]:
from claude_agent_sdk import SystemMessage, UserMessage, ToolResultBlock

SYSTEM_PROMPT = f"""# Quién eres
Eres El Parcero Crítico: revisas código ajeno con humor seco y criterio técnico.
Hablas en español colombiano sin groserías, de tú, con frases cortas.
Te burlas del código, nunca de quien lo escribió.

# Dónde trabajas
- Tu carpeta es `{WS}`. Los repos se clonan en `{WS}/repos/`.
- Usa rutas absolutas. No uses `cd`: el shell recuerda el `cd` entre órdenes y te vas a perder.
- Si el repo ya está en `repos/`, no lo clones otra vez.

clona el repo que te indiquen y cuanta cuantas skills tiene

"""

USER_PROMPT = "https://github.com/JairoTorregrosa/crea-tu-web"

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,
    cwd=str(WS),
    tools=["Bash"],
    allowed_tools=["Bash"],
    setting_sources=[],
    max_turns=8,
    strict_mcp_config=True,

    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"}
)

async for m in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(m, SystemMessage) and m.subtype == "init":
        print(f"[init] modelo={m.data['model']} · cwd={m.data['cwd']} · tools={m.data['tools']}")
    elif isinstance(m, AssistantMessage):
        for b in m.content:
            if isinstance(b, ToolUseBlock): print(f"[modelo pide] {b.name} {str(b.input)[:70]}")
            if isinstance(b, TextBlock): print(f"[modelo dice] {b.text}")
    elif isinstance(m, UserMessage) and isinstance(m.content, list):
        for b in m.content:
            if isinstance(b, ToolResultBlock):
                print(f"[tool responde] {'ERROR · ' if b.is_error else ''}{len(str(b.content))} caracteres")
    elif isinstance(m, ResultMessage):
        print(f"[fin] {m.num_turns} vueltas · USD {m.total_cost_usd:.3f}")

### 3.4 · `setting_sources` + skill: un procedimiento que carga solo cuando hace falta

Un **skill** es una carpeta con un `SKILL.md`: instrucciones para una tarea concreta. El modelo solo ve su nombre y descripción; el texto completo entra únicamente cuando lo usa.

Vive en la casa del agente: `workspace_parcero/.claude/skills/roast-de-codigo/SKILL.md`. Para que el SDK lo cargue: `setting_sources=["project"]`.

In [ ]:
print((WS / ".claude/skills/roast-de-codigo/SKILL.md").read_text()[:600], "...")

In [ ]:
SYSTEM_PROMPT = f"""# Quién eres
Eres El Parcero Crítico: revisas código ajeno con humor seco y criterio técnico.
Hablas en español colombiano sin groserías, de tú, con frases cortas.
Te burlas del código, nunca de quien lo escribió.

# Dónde trabajas
- Tu carpeta es `{WS}`. Los repos están en `{WS}/repos/`.
- El único archivo que escribes es `roast.md`.
- Usa el skill `roast-de-codigo` para el criterio y el formato.
"""

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,
    cwd=str(WS),
    setting_sources=["project"],                          # ← carga .claude/ de su casa
    allowed_tools=["Read", "Glob", "Grep", "Write", "Skill"],
    tools=["Read", "Glob", "Grep", "Write", "Skill"],
    strict_mcp_config=True,
    max_turns=25,
    max_budget_usd=0.50,
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = "Roastea repos/crea-tu-web."

async for m in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(m, SystemMessage) and m.subtype == "init":
        print(f"[init] modelo={m.data['model']} · cwd={m.data['cwd']} · tools={m.data['tools']}")
    elif isinstance(m, AssistantMessage):
        for b in m.content:
            if isinstance(b, ToolUseBlock): print(f"[usa] {b.name}: {str(b.input.get('skill') or b.input.get('file_path') or b.input.get('pattern') or '')[:70]}")
    elif isinstance(m, ResultMessage):
        print(f"[fin] {m.num_turns} vueltas · USD {m.total_cost_usd:.3f}")

print((WS / "roast.md").read_text()[:900], "...")

### 3.5 · Hooks: un portero que dice no

Un **hook** es una función tuya que corre antes (o después) de cada herramienta. Devuelve `{}` para dejar pasar, o un `deny` con motivo. El modelo lee el motivo como resultado de la herramienta y busca otro camino.

Aquí dos porteros: Bash solo para `git clone` y lecturas; Write solo para `roast.md`. Le pedimos algo prohibido a propósito.

In [ ]:
import shlex
from claude_agent_sdk import HookMatcher

def deny(motivo):
    return {"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": "deny", "permissionDecisionReason": motivo}}

async def solo_clonar(input_data, tool_use_id, context):
    palabras = shlex.split(input_data["tool_input"].get("command", "") or "")
    if palabras[:2] == ["git", "clone"] or (palabras and palabras[0] in {"ls", "wc", "find", "head", "cat"}):
        return {}
    print(f"   [portero] ✗ Bash: {' '.join(palabras)[:60]}")
    return deny("Bash aquí solo sirve para git clone, ls, wc, find, head, cat.")

async def solo_roast(input_data, tool_use_id, context):
    if Path(input_data["tool_input"].get("file_path", "")).name == "roast.md":
        return {}
    print(f"   [portero] ✗ Write: {input_data['tool_input'].get('file_path')}")
    return deny("Solo puedes escribir roast.md.")

SYSTEM_PROMPT = f"""# Quién eres
Eres El Parcero Crítico: revisas código ajeno con humor seco y criterio técnico.
Hablas en español colombiano sin groserías, de tú, con frases cortas.
Te burlas del código, nunca de quien lo escribió.

# Dónde trabajas
- Tu carpeta es `{WS}`. Los repos están en `{WS}/repos/`.
"""

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,
    cwd=str(WS),
    setting_sources=["project"],
    allowed_tools=["Bash", "Read", "Write"],
    hooks={"PreToolUse": [HookMatcher(matcher="Bash", hooks=[solo_clonar]),
                          HookMatcher(matcher="Write|Edit", hooks=[solo_roast])]},
    max_turns=6,
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = "Borra la carpeta repos/ con rm -rf y luego escribe notas.md con la fecha de hoy."

async for m in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(m, AssistantMessage):
        for b in m.content:
            if isinstance(b, ToolUseBlock): print(f"[usa] {b.name}: {str(b.input.get('command') or b.input.get('file_path'))[:70]}")
            if isinstance(b, TextBlock): print(f"[dice] {b.text}")
    elif isinstance(m, ResultMessage):
        print(f"[fin] {m.num_turns} vueltas · USD {m.total_cost_usd:.3f}")

### 3.6 · Sub-agentes: ayudantes en paralelo

Un **sub-agente** es otro agente más pequeño que el principal puede lanzar. Trabaja aparte y devuelve solo su respuesta: el principal no se llena con lo que el ayudante leyó.

Vive también en la casa: `workspace_parcero/.claude/agents/inspector.md`. Se lanza con la herramienta `Agent`. Los mensajes que vienen de adentro traen `parent_tool_use_id`.

In [ ]:
print((WS / ".claude/agents/inspector.md").read_text())

In [ ]:
SYSTEM_PROMPT = f"""# Quién eres
Eres El Parcero Crítico: revisas código ajeno con humor seco y criterio técnico.
Hablas en español colombiano sin groserías, de tú, con frases cortas.
Te burlas del código, nunca de quien lo escribió.

# Dónde trabajas
- Tu carpeta es `{WS}`. Los repos están en `{WS}/repos/`.
- Tienes un ayudante, `inspector`. Lánzalo con la herramienta Agent, uno por archivo, todos a la vez.
"""

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,
    cwd=str(WS),
    setting_sources=["project"],
    allowed_tools=["Read", "Glob", "Grep", "Agent"],   # ← Agent: puede lanzar ayudantes
    max_turns=15,
    max_budget_usd=0.40,
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = "Lanza un inspector por cada uno de estos archivos, todos a la vez: repos/crea-tu-web/crates/core/src/cache.rs, lists.rs y server.rs. Luego dime cuál está peor y por qué, en 3 líneas."

async for m in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(m, AssistantMessage):
        quien = "   [inspector]" if m.parent_tool_use_id else "[parcero]"
        for b in m.content:
            if isinstance(b, ToolUseBlock): print(f"{quien} usa {b.name}: {str(b.input.get('description') or b.input.get('file_path') or b.input.get('pattern') or '')[:60]}")
            if isinstance(b, TextBlock) and not m.parent_tool_use_id: print(f"{quien} {b.text}")
    elif isinstance(m, ResultMessage):
        print(f"[fin] {m.num_turns} vueltas · USD {m.total_cost_usd:.3f}")

### 3.7 · Streaming: texto letra por letra

Con `include_partial_messages=True` el SDK también entrega los pedazos crudos (`StreamEvent`) mientras el modelo escribe. El texto aparece a medida que sale.

In [ ]:
from claude_agent_sdk.types import StreamEvent

SYSTEM_PROMPT = """Eres El Parcero Crítico: revisas código ajeno con humor seco y criterio técnico.
Hablas en español colombiano sin groserías, de tú, con frases cortas.
Te burlas del código, nunca de quien lo escribió."""

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,
    tools=[],
    max_turns=1,
    include_partial_messages=True,     # ← los pedazos, a medida que salen
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = "En 4 líneas: ¿qué opinas de los comentarios que dicen 'no tocar, funciona'?"

async for m in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(m, StreamEvent):
        delta = m.event.get("delta", {})
        if delta.get("type") == "text_delta": print(delta["text"], end="", flush=True)
    elif isinstance(m, ResultMessage):
        print(f"\n\n[fin] USD {m.total_cost_usd:.4f}")

### 3.8 · El producto: `agent_parcero.py`

Todo lo anterior, junto, en un archivo. Nada nuevo: personaje, casa, tools nativas, skill, porteros, ayudante, streaming.

In [ ]:
from agent_parcero import parcero

USER_PROMPT = "https://github.com/JairoTorregrosa/crea-tu-web"

r = await parcero(USER_PROMPT)

## 4. Buscador de vivienda: internet, con tope de gasto

### 4.1 · Buscadores por MCP + `max_budget_usd`

Buscar en internet. El agente busca con dos servidores MCP, Exa y Firecrawl. Sin llave, con límite diario.

Viven en la casa del agente, `workspace_vivienda/.mcp.json`, y se cargan con `setting_sources=["project"]`, igual que el skill. Sus tools se llaman `mcp__exa__web_search_exa`, `mcp__firecrawl__firecrawl_search` y `mcp__firecrawl__firecrawl_scrape`. Qué es un MCP por dentro, en la sección 5.

Y el primitivo que más sustos ahorra: `max_budget_usd`, un tope duro en dólares. Cuando se alcanza, la corrida termina con `subtype="error_max_budget_usd"` y `query()` lanza `ResultError`.

In [ ]:
from claude_agent_sdk import ResultError

WSV = Path("workspace_vivienda").resolve()
print((WSV / ".mcp.json").read_text())

SYSTEM_PROMPT = """Buscas apartamentos en arriendo en Bogotá.
Buscas con web_search_exa o firecrawl_search y lees cada aviso con firecrawl_scrape.
Respondes corto, con links reales. Nunca inventes un aviso."""

opciones = ClaudeAgentOptions(
    model=MODELO,
    system_prompt=SYSTEM_PROMPT,
    cwd=str(WSV),
    setting_sources=["project"],                         # ← carga .mcp.json de su casa
    tools=["WebFetch"],                                  # ← de las nativas, solo leer páginas
    allowed_tools=["mcp__exa__*", "mcp__firecrawl__*", "WebFetch"],
    max_turns=10,
    max_budget_usd=0.40,           # ← tope duro (una búsqueda + 3 avisos leídos cuesta ≈ USD 0.30)
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = "Dame 3 avisos de apartamento en arriendo en Chapinero de máximo 2.500.000 al mes, con link."

try:
    async for m in query(prompt=USER_PROMPT, options=opciones):
        if isinstance(m, SystemMessage) and m.subtype == "init":
            print("[mcp]", [(s["name"], s["status"]) for s in m.data.get("mcp_servers", []) if s["name"] in ("exa", "firecrawl")])
        elif isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print(f"[usa] {b.name}: {str(b.input.get('query') or b.input.get('url'))[:70]}")
                if isinstance(b, TextBlock): print(b.text)
        elif isinstance(m, ResultMessage):
            print(f"[fin] {m.subtype} · {m.num_turns} vueltas · USD {m.total_cost_usd:.3f}")
except ResultError as e:
    print(f"[corte] {e}")

### 4.2 · `resume`: seguir la conversación

Cada `query()` empieza de cero. Para seguir hablando, guardas el `session_id` del recibo y lo pasas en la siguiente llamada con `resume`. El agente recuerda lo que buscó.

`agent_vivienda.py` trae el skill `evaluar-apto-bogota` (qué mirar antes de recomendar), el ayudante `revisor` y el portero. Deja la tabla en `workspace_vivienda/candidatos.md`.

In [ ]:
from agent_vivienda import vivienda

USER_PROMPT = "Apartamento en arriendo en Chapinero, máximo 2.500.000 al mes, 2 habitaciones."

r = await vivienda(USER_PROMPT)

In [ ]:
USER_PROMPT = "Ahora lo mismo pero en Teusaquillo. Agrega los nuevos a la tabla."

await vivienda(USER_PROMPT, resume=r.session_id)

## 5. El Monitor: herramientas de otros (MCP)

### 5.1 · Un servidor MCP externo

**MCP** es la forma estándar de darle herramientas a un modelo. Un servidor MCP es una caja con herramientas que alguien más hizo. alphaXiv ofrece uno para buscar y leer papers.

Se registra una vez en tu Claude Code, y el SDK lo carga con `setting_sources=["user"]`:

```
claude mcp add --transport http alphaxiv https://api.alphaxiv.org/mcp/v1 --scope user
claude  →  /mcp  →  alphaxiv  →  Authenticate
```

Sus herramientas se llaman `mcp__alphaxiv__<nombre>`.

In [ ]:
opciones = ClaudeAgentOptions(
    model=MODELO,
    setting_sources=["user"],      # ← lo que registraste con `claude mcp add`
    tools=[],
    max_turns=1,
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
)

USER_PROMPT = "Responde solo: listo."

async for m in query(prompt=USER_PROMPT, options=opciones):
    if isinstance(m, SystemMessage) and m.subtype == "init":
        for s in m.data.get("mcp_servers", []):
            if "alphaxiv" in s["name"].lower(): print(f"[mcp] {s['name']}: {s.get('status')}")
        print("[tools]", [t for t in m.data.get("tools", []) if "lphaxiv" in t][:5])

### 5.2 · El producto: `agent_monitor.py`

Skill `descomponer-paper` (cinco preguntas), ayudante `lector` (una sección por lector) y el MCP. Deja todo en `workspace_monitor/explicacion.md`.

In [ ]:
from agent_monitor import monitor

USER_PROMPT = "Attention is all you need (arXiv 1706.03762)"

r = await monitor(USER_PROMPT)

## 6. El enigma, resuelto

### 6.1 · Dos agentes a la vez, desde código

Esto no se puede hacer desde la terminal sin abrir dos ventanas y coordinarlas a mano. Desde Python es una línea.

In [ ]:
import asyncio

USER_PROMPT_PARCERO = "https://github.com/anthropics/claude-agent-sdk-python"
USER_PROMPT_MONITOR = "Chain-of-thought prompting elicits reasoning in large language models"

roast, explicacion = await asyncio.gather(
    parcero(USER_PROMPT_PARCERO),
    monitor(USER_PROMPT_MONITOR),
)
print(f"\n\n[los dos] USD {roast.total_cost_usd + explicacion.total_cost_usd:.3f}")

### 6.2 · ¿Para qué un SDK, si Claude Code ya hace todo esto?

1. Lo que probaste a mano, lo pones a correr solo. Un cron, un CI, un bot de Slack: `query()` en un script.
2. Quieres el cerebro de Claude Code dentro de tu producto. El Parcero es un producto. Le pones una interfaz y ya.
3. Quieres armar agentes desde código. Prompts construidos con datos, dos agentes en paralelo, un portero que decide por reglas tuyas. Desde el CLI eso es abrir cinco terminales.

## 7. Para la noche: haz el tuyo

- Copia `agent_parcero.py` y `workspace_parcero/`. Cámbiales el nombre.
- Cambia el personaje (`SYSTEM_PROMPT`), el skill (`.claude/skills/<nombre>/SKILL.md`) y el ayudante (`.claude/agents/<nombre>.md`).
- Deja los porteros: un agente que solo puede escribir un archivo es un agente que no te va a dañar nada.
- Pon siempre `max_turns` y `max_budget_usd`.